# Design exercise D4 · Bike-share demand — Part B

RomaPedala's operations team fixes the van rota for the coming week on
**Sunday evening**. Your job: forecast the rentals at each of 30 stations for
every hour of the week from Monday 1 to Sunday 7 June 2026, 5,040 numbers in
all, as close to what really happens as you can.

You have about 15 minutes. The cells below load the data, define the metric,
backtest a baseline over eight past weeks and write the file. Make **your**
design choices in the cell marked *Your model*, compare them in the backtest,
then hand in.

**Scoring.** Your forecasts are compared with the rentals that really
happened, which only the grader has. The mean absolute error, in bikes, gives
the points: 0 at 10.3 (each station's average rentals, every hour), 10 at our
reference forecast's 3.56 or below, linear in between.

> **One attempt.** Only the **first** file you turn in is scored. If you
> unsubmit and turn in another file, it is not scored: you keep the score of
> the first. Choose your model with the notebook's own validation, check the file with the last
> cell, and only then turn it in.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression

BASE = "https://raw.githubusercontent.com/fabsilvestri/AIML-Course/main/design/d04/"
train = pd.read_csv(BASE + "train.csv")    # 21 weeks, one row per station and hour
new = pd.read_csv(BASE + "test.csv")       # the week to forecast, no rentals
for frame in (train, new):
    frame["time"] = pd.to_datetime(frame["timestamp"])

assert train.shape == (105840, 5) and new.shape == (5040, 5)
assert train["station"].nunique() == new["station"].nunique() == 30
assert train["time"].max() < new["time"].min()
print(f"{len(train):,} station-hours from {train['time'].min():%d %b} to "
      f"{train['time'].max():%d %b}; {len(new):,} to forecast")
train.head()

The metric is the mean absolute error, in bikes. The backtest is a
**rolling origin**: at each of eight Monday 00:00 cut-offs from 6 April, the
forecast sees only the rows before the cut-off — what the team has on Sunday
evening — and forecasts the 168 hours after it. Copying last week is scored on
the same hours beside it.

A forecast is a function `forecast(history, future)`: `history` has every
column, `future` has no `rentals`, and it returns one number per row of
`future`, in its order.

In [ ]:
def mae(truth, pred):
    return float(np.mean(np.abs(np.asarray(truth) - np.asarray(pred))))

def weeks_ago(history, future, k):
    """For each row of `future`: the rentals at the same station and the same
    hour k weeks earlier, looked up in `history` (NaN if not there)."""
    past = history[["station", "time", "rentals"]].copy()
    past["time"] = past["time"] + pd.Timedelta(weeks=k)
    out = future[["station", "time"]].merge(past, on=["station", "time"],
                                            how="left")
    return out["rentals"].to_numpy(dtype=float)

def copy_last_week(history, future):
    return weeks_ago(history, future, 1)

CUTS = pd.date_range("2026-04-06", periods=8, freq="7D")     # Mondays, 00:00

def backtest(forecast):
    """Forecast each of the eight weeks from the rows before it."""
    rows = []
    for cut in CUTS:
        history = train[train["time"] < cut]
        week = train[(train["time"] >= cut)
                     & (train["time"] < cut + pd.Timedelta(weeks=1))]
        future = week.drop(columns="rentals").reset_index(drop=True)
        pred = np.asarray(forecast(history, future), dtype=float)
        assert pred.shape == (len(future),) and not np.isnan(pred).any()
        rows.append({"week from": f"{cut:%d %b}",
                     "forecast": mae(week["rentals"], pred),
                     "copy last week": mae(week["rentals"],
                                           copy_last_week(history, future))})
    table = pd.DataFrame(rows).set_index("week from").round(2)
    print(table.T.to_string())
    print(f"mean: forecast {table['forecast'].mean():.3f}, "
          f"copy last week {table['copy last week'].mean():.3f}")
    return table["forecast"].mean()

A baseline, to beat: copy the same hour of the week before.

In [ ]:
baseline = backtest(copy_last_week)

## Your model

Write your forecast here and compare it with the baseline in the backtest.
Every choice — which past values, which columns, whether to fit a model and
which — is yours, and is judged by `backtest`. Whatever you fit, fit it
inside `forecast`, on `history` only.

In [ ]:
def my_forecast(history, future):
    return copy_last_week(history, future)     # replace with your design

score = backtest(my_forecast)

## Hand in

Forecast the week from all 21 weeks of history, and check the file before you
write it.

In [ ]:
pred = np.asarray(my_forecast(train, new.copy()), dtype=float)

out = pd.DataFrame({"id": new["id"], "rentals": pred})
assert len(out) == 5040 and out["id"].is_unique
assert set(out["id"]) == set(new["id"])
assert out["rentals"].notna().all() and np.isfinite(out["rentals"]).all()
out.to_csv("forecast.csv", index=False)
print("wrote forecast.csv:", len(out), "rows")

In Colab, download it (the folder icon on the left, or run the next cell),
then in Classroom open the assignment, **Add → File**, choose
`forecast.csv`, and **Turn in**. The grader marks it within the hour.
**Only this first file counts**: a second one is not scored.

In [ ]:
try:
    from google.colab import files
    files.download("forecast.csv")
except ImportError:
    print("Not in Colab: forecast.csv is in the working directory.")